# SKENARIO 3 — CoT vs non-CoT (fine-tuned)

Bandingkan **model fine-tune CoT terbaik** vs **model fine-tune non-CoT** di test set yang SAMA.
Base + hyperparam identik, beda **cuma data training (cot vs nocot)** -> selisih = efek CoT.

**Sampling** (N=3 kandidat/soal, temp 0.7), ekstrak `\boxed{}`, cocokkan ke gold. Butuh **GPU**.
Metrik: **pass@1, pass@2, pass@3** + delta (CoT − nonCoT) per metrik/set.

## 0. Setup + install peft

In [ ]:
!pip install -q -U peft transformers accelerate

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = '/kaggle/working/FP_NLP'
URL  = 'https://github.com/henray404/FP_NLP.git'
if os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'fetch','-q','origin','main'], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','-q',URL,REPO], check=True)
sys.path.insert(0, REPO); os.chdir(REPO)
print('repo ready:', REPO)

## 1. Config — base, 2 adapter (cot + nocot), test set

Add Input: dataset berisi `adapter_cot_1.5b/` + `adapter_nocot_1.5b/` (hasil training).

In [ ]:
import glob
BASE = 'Qwen/Qwen2.5-1.5B'

def find_adapter(keyword):
    hits = glob.glob(f'/kaggle/input/**/*{keyword}*/adapter_config.json', recursive=True)
    return str(Path(hits[0]).parent) if hits else None

ADAPTER_COT   = find_adapter('adapter_cot')
ADAPTER_NOCOT = find_adapter('adapter_nocot')
assert ADAPTER_COT and ADAPTER_NOCOT, f'adapter kurang: cot={ADAPTER_COT} nocot={ADAPTER_NOCOT}'
print('CoT  :', ADAPTER_COT)
print('nonCoT:', ADAPTER_NOCOT)

def find_set(keyword):
    for pat in [f'/kaggle/input/**/*{keyword}*test*.jsonl', f'data/sft/test/*{keyword}*.jsonl']:
        hits = glob.glob(pat, recursive=True)
        if hits: return hits[0]
    return f'data/sft/test/{keyword}_test.jsonl'

SET_PATHS = {'numglue': find_set('numglue'), 'un': find_set('un'), 'easy': find_set('easy')}
print('test sets:', SET_PATHS)

## 2. Eval CoT vs non-CoT + tabel

In [ ]:
from src.eval.scenario_eval import load_sets
from src.eval.sample_eval import eval_specs_sampling
from src.eval.sampling_metrics import render_tables
import json

sets = load_sets(SET_PATHS)
specs = {
    'CoT':    {'model': BASE, 'adapter': ADAPTER_COT},
    'nonCoT': {'model': BASE, 'adapter': ADAPTER_NOCOT},
}
METRICS = ['pass@1', 'pass@2', 'pass@3']

# 3 sampel/soal (temp 0.7) cukup utk pass@3. maj tak dipakai di S3.
results = eval_specs_sampling(specs, sets, n_samples=3, ks_pass=(1, 2, 3), ks_maj=(),
                             temperature=0.7, top_p=0.95, max_new_tokens=2048, batch_size=4)

print('\n=== SKENARIO 3: CoT vs non-CoT (sampling) ===')
print(render_tables(results, METRICS))

# selisih (CoT - nonCoT) tiap metrik tiap set
print('\ndelta (CoT - nonCoT):')
for m in METRICS:
    for s in sorted(sets):
        c = results['CoT'][s][m]; n = results['nonCoT'][s][m]
        print(f'  {m:7} {s:10} CoT={c:.3f} nonCoT={n:.3f} delta={c-n:+.3f}')

Path('data/eval').mkdir(parents=True, exist_ok=True)
Path('data/eval/s3_cot_vs_noncot.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
print('\nsummary -> data/eval/s3_cot_vs_noncot.json')